# Naive Bayes Classifier

The **Naive Bayes classifier** is one of the most elegant and practically effective algorithms in machine learning. Despite its simplicity and a seemingly unrealistic assumption (feature independence), it consistently delivers strong results, especially on text classification, spam detection, and medical diagnosis tasks.

In these notes, we will cover:
1. **Bayes' Theorem:** The probabilistic foundation (Prior, Likelihood, Evidence, Posterior).
2. **The Naive Bayes Classifier:** Full mathematical derivation and the "Naive" conditional independence assumption.
3. **Types of Naive Bayes:** Gaussian, Multinomial, and Bernoulli variants.
4. **Laplace Smoothing:** Handling zero-probability edge cases.
5. **Advantages, Disadvantages & Practical Use Cases.**
6. **Python Implementations:** Hands-on code using Scikit-Learn.

## 1. Bayes' Theorem: The Foundation

Before understanding Naive Bayes, we need to understand **Bayes' Theorem**, which provides a principled way to update our beliefs about a hypothesis (class label) given observed evidence (input features).

### The Formula

$$P(A | B) = \frac{P(B | A) \cdot P(A)}{P(B)}$$

In the context of classification, let $C$ be a class label and $X$ be the feature vector:

$$P(C | X) = \frac{P(X | C) \cdot P(C)}{P(X)}$$

### Components Explained

| Term | Name | Meaning |
| :--- | :--- | :--- |
| $P(C)$ | **Prior Probability** | The probability of the class *before* observing any features. Estimated from training data as the relative frequency of each class. |
| $P(X \| C)$ | **Likelihood** | The probability of observing the feature values $X$ *given* that the sample belongs to class $C$. This is the core modeling component. |
| $P(X)$ | **Evidence (Marginal Likelihood)** | The total probability of observing the feature values $X$ across all classes. Acts as a normalization constant. |
| $P(C \| X)$ | **Posterior Probability** | The probability of class $C$ *after* observing the features $X$. This is what we want to compute for classification. |

### Intuitive Example: Medical Diagnosis
Suppose we want to classify whether a patient has a disease ($C = \text{Disease}$) based on a positive test result ($X = \text{Positive}$).

- **Prior ($P(C)$):** Only $1\%$ of the population has the disease. So $P(\text{Disease}) = 0.01$.
- **Likelihood ($P(X|C)$):** The test correctly detects the disease $90\%$ of the time. So $P(\text{Positive} | \text{Disease}) = 0.90$.
- **False Positive Rate:** The test incorrectly gives a positive result $5\%$ of the time for healthy people. So $P(\text{Positive} | \text{No Disease}) = 0.05$.

Using Bayes' Theorem:
$$P(\text{Disease} | \text{Positive}) = \frac{0.90 \times 0.01}{(0.90 \times 0.01) + (0.05 \times 0.99)} = \frac{0.009}{0.009 + 0.0495} = \frac{0.009}{0.0585} \approx 0.154$$

Despite a $90\%$ accurate test, a positive result only indicates a $15.4\%$ probability of actual disease! This is because the prior probability of the disease is very low. **Bayes' Theorem elegantly incorporates this prior knowledge.**

## 2. The Naive Bayes Classifier: Derivation

### From Bayes' Theorem to a Classifier
For a classification problem with $K$ classes and a feature vector $X = [x_1, x_2, \dots, x_d]$, we want to find the class $C_k$ that maximizes the posterior probability:

$$\hat{y} = \arg\max_{C_k} P(C_k | X) = \arg\max_{C_k} \frac{P(X | C_k) \cdot P(C_k)}{P(X)}$$

Since $P(X)$ is the same for all classes (it's a constant denominator), we can simplify:

$$\hat{y} = \arg\max_{C_k} P(X | C_k) \cdot P(C_k)$$

### The Problem: Estimating $P(X | C_k)$
Here $X = [x_1, x_2, \dots, x_d]$ is a $d$-dimensional feature vector. Computing the joint probability $P(x_1, x_2, \dots, x_d | C_k)$ requires an exponentially large amount of data (for continuous features) or a combinatorially explosive lookup table (for discrete features). This is computationally infeasible for real-world datasets.

### The "Naive" Assumption: Conditional Independence
To make the computation tractable, Naive Bayes introduces a **strong simplifying assumption**: all features are **conditionally independent** given the class label.

$$P(x_1, x_2, \dots, x_d | C_k) = P(x_1 | C_k) \cdot P(x_2 | C_k) \cdot \dots \cdot P(x_d | C_k) = \prod_{j=1}^{d} P(x_j | C_k)$$

#### Why is this "Naive"?
In the real world, features are almost **never** truly independent. For example:
- In text classification, the word "machine" and "learning" are highly correlated.
- In medical data, blood pressure and cholesterol are correlated.

Despite violating this assumption, Naive Bayes works surprisingly well in practice because:
1. Even if individual probability estimates are biased, the **ranking** of posterior probabilities across classes often remains correct.
2. The decision boundary is determined by which class has the *highest* posterior, not the exact probability values.

### The Final Naive Bayes Classification Rule

Substituting the independence assumption:

$$\hat{y} = \arg\max_{C_k} P(C_k) \prod_{j=1}^{d} P(x_j | C_k)$$

#### Log Transformation (Practical Computation)
Multiplying many small probabilities leads to **floating-point underflow**. To prevent this, we take the logarithm (converting products into sums):

$$\hat{y} = \arg\max_{C_k} \left[ \log P(C_k) + \sum_{j=1}^{d} \log P(x_j | C_k) \right]$$

This is the form used by all practical implementations of Naive Bayes.

### The Training Process
Training a Naive Bayes model is essentially **counting and estimating distributions** from the training data:

1. **Estimate Prior Probabilities $P(C_k)$:** Count the proportion of each class in the training set.
   $$P(C_k) = \frac{\text{Number of samples in class } C_k}{\text{Total number of samples}}$$
2. **Estimate Likelihoods $P(x_j | C_k)$:** For each feature $j$ and each class $k$, estimate the conditional probability distribution.
   - **Discrete features:** Count the frequency of each feature value within each class.
   - **Continuous features:** Fit a probability distribution (commonly Gaussian/Normal) to the feature values within each class.

This is why Naive Bayes is **extremely fast** to train—there is no iterative optimization, no gradient descent, just statistical counting!

## 3. Gaussian Naive Bayes

**Gaussian Naive Bayes (GaussianNB)** is used when the input features are **continuous** (real-valued numbers). It assumes that the values of each feature, within each class, follow a **Gaussian (Normal) distribution**.

### The Likelihood Formula
For a continuous feature $x_j$ given class $C_k$:

$$P(x_j | C_k) = \frac{1}{\sqrt{2 \pi \sigma_{jk}^2}} \exp \left( -\frac{(x_j - \mu_{jk})^2}{2 \sigma_{jk}^2} \right)$$

Where:
- $\mu_{jk}$ = Mean of feature $j$ for all samples in class $C_k$.
- $\sigma_{jk}^2$ = Variance of feature $j$ for all samples in class $C_k$.

### Training
For each feature $j$ and each class $k$, we simply compute the **sample mean** ($\mu_{jk}$) and **sample variance** ($\sigma_{jk}^2$) from the training data.

### When to Use
- Continuous numerical feature data (e.g., height, weight, temperature, sensor readings).
- Works well when the Gaussian assumption is reasonably met (bell-shaped feature distributions within each class).

### Worked Example: Gaussian NB by Hand
Suppose we have a dataset with 2 features ($x_1$, $x_2$) and 2 classes (Spam / Not Spam). After training:

| Parameter | Spam | Not Spam |
| :--- | :---: | :---: |
| $P(C_k)$ (Prior) | $0.4$ | $0.6$ |
| $\mu_1, \sigma_1$ (Feature 1) | $5.0, 1.0$ | $2.0, 1.5$ |
| $\mu_2, \sigma_2$ (Feature 2) | $3.0, 0.5$ | $7.0, 1.0$ |

For a new sample $X = [4.5, 3.2]$, we:
1. Compute $P(x_1 = 4.5 | \text{Spam})$ using the Gaussian PDF with $\mu=5.0, \sigma=1.0$.
2. Compute $P(x_2 = 3.2 | \text{Spam})$ using the Gaussian PDF with $\mu=3.0, \sigma=0.5$.
3. Multiply: $P(\text{Spam}) \times P(x_1|\text{Spam}) \times P(x_2|\text{Spam})$.
4. Repeat for "Not Spam".
5. Predict the class with the higher score.

## 4. Multinomial Naive Bayes

**Multinomial Naive Bayes (MultinomialNB)** is designed for **discrete count-based features**, most commonly used for **text classification** where features represent word frequencies or term counts.

### The Likelihood Formula
For a feature $x_j$ (representing the count/frequency of word $j$) given class $C_k$:

$$P(x_j | C_k) = \frac{N_{jk}}{N_k}$$

Where:
- $N_{jk}$ = Total count of feature $j$ (e.g., occurrences of word $j$) across all documents in class $C_k$.
- $N_k$ = Total count of all features (total words) across all documents in class $C_k$.

### When to Use
- **Text Classification:** Spam detection, sentiment analysis, topic categorization.
- **Document Classification:** When features are word counts (Bag of Words) or TF-IDF scores.
- Feature values must be **non-negative integers or floats** representing counts/frequencies.

## 5. Bernoulli Naive Bayes

**Bernoulli Naive Bayes (BernoulliNB)** is designed for **binary/boolean features** (features that are either $0$ or $1$, indicating the absence or presence of a characteristic).

### The Likelihood Formula
For a binary feature $x_j \in \{0, 1\}$ given class $C_k$:

$$P(x_j | C_k) = P(j | C_k)^{x_j} \cdot (1 - P(j | C_k))^{(1 - x_j)}$$

Where $P(j | C_k)$ is the probability of feature $j$ being present (equal to $1$) in class $C_k$.

### Key Difference from Multinomial NB
- **Multinomial NB** considers how *many times* a word appears (count matters).
- **Bernoulli NB** only considers *whether* a word appears or not (binary presence).
- Bernoulli NB explicitly models the **absence** of features (penalizes if an expected feature is missing), whereas Multinomial NB simply ignores absent features.

### When to Use
- Short text documents where word presence/absence is more informative than frequency.
- Binary feature datasets (e.g., survey responses: Yes/No, feature flags: True/False).

### Comparison Table: Naive Bayes Variants

| Variant | Feature Type | Likelihood Model | Best Use Case |
| :--- | :--- | :--- | :--- |
| **GaussianNB** | Continuous (real-valued) | Gaussian PDF ($\mu, \sigma^2$) | Numerical data (Iris, sensor data, medical measurements) |
| **MultinomialNB** | Discrete counts / frequencies | Multinomial Distribution (word counts) | Text classification (spam, sentiment), document categorization |
| **BernoulliNB** | Binary ($0$ or $1$) | Bernoulli Distribution (presence/absence) | Short-text classification, binary survey features |

## 6. Laplace Smoothing (Additive Smoothing)

### The Zero-Probability Problem
Consider a scenario in text classification where a word appears in the test data but **never appeared** in the training data for a specific class. The likelihood $P(x_j | C_k) = 0$.

Since we multiply likelihoods:
$$P(C_k) \prod_{j=1}^{d} P(x_j | C_k) = 0$$

A single zero probability **wipes out the entire posterior**, regardless of how strong all other feature evidences are. This is catastrophic.

### The Solution: Laplace Smoothing
To prevent zero probabilities, we add a small smoothing parameter $\alpha$ (usually $\alpha = 1$, called Laplace Smoothing) to all feature counts:

$$P(x_j | C_k) = \frac{N_{jk} + \alpha}{N_k + \alpha \cdot d}$$

Where:
- $N_{jk}$ = Count of feature $j$ in class $C_k$.
- $N_k$ = Total count of all features in class $C_k$.
- $d$ = Total number of distinct features (vocabulary size for text).
- $\alpha$ = Smoothing parameter (hyperparameter, default $1$).

### Effect
- Ensures no probability is ever exactly zero.
- When $\alpha = 1$ (**Laplace Smoothing**): Adds $1$ pseudo-count to each feature.
- When $\alpha < 1$ (**Lidstone Smoothing**): Adds a fractional pseudo-count.
- When $\alpha = 0$: No smoothing (original formula, risk of zero probabilities).
- In Scikit-Learn, this is controlled via the `alpha` hyperparameter in `MultinomialNB` and `BernoulliNB`.

## 7. Advantages, Disadvantages & When to Use Naive Bayes

### Advantages
1. **Extremely Fast:** Training is just counting/computing statistics — no iterative optimization (gradient descent) needed. Both training and prediction are $O(n \cdot d)$.
2. **Works Well with High-Dimensional Data:** Excellent performance on text data with thousands of features (words), where other algorithms struggle.
3. **Handles Small Datasets Well:** Because it estimates simple statistics (means, variances, counts) rather than complex decision boundaries, it generalizes well even with limited data.
4. **Robust to Irrelevant Features:** Irrelevant features contribute roughly equal likelihoods across all classes, effectively canceling out in the argmax decision.
5. **Naturally Handles Multi-Class:** Directly computes posteriors for all $K$ classes without needing One-vs-Rest or One-vs-One wrappers.
6. **Probabilistic Output:** Produces class probability estimates, not just hard labels.

### Disadvantages
1. **Conditional Independence Assumption:** Features are rarely truly independent. Correlated features can degrade probability estimates (though rankings often remain useful).
2. **Poor Probability Calibration:** While the predicted class is often correct, the actual probability values tend to be pushed toward $0$ and $1$ (overconfident). Calibration techniques (e.g., Platt Scaling) may be needed.
3. **Zero-Frequency Problem:** Without Laplace Smoothing, unseen feature values result in zero posterior probabilities.
4. **Assumption of Feature Distribution:** Gaussian NB assumes normality; if the true distribution is heavily skewed or multimodal, performance suffers.

### When to Use
- **Text classification** (spam, sentiment analysis, topic modeling) — primary use case.
- **Baseline model** for any classification problem.
- **Real-time prediction** systems where speed is critical.
- **Small training sets** where complex models would overfit.
- **High-dimensional sparse data** (e.g., Bag of Words with 10,000+ features).

## 8. Code Setup & Imports

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.naive_bayes import GaussianNB, MultinomialNB, BernoulliNB
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

# Set visualization contexts
plt.style.use('seaborn-v0_8-whitegrid')
sns.set_context('notebook', font_scale=1.1)

import warnings
warnings.filterwarnings('ignore')

print("Libraries imported successfully!")

## 9. Demo A: Gaussian Naive Bayes on the Iris Dataset

We will train a Gaussian NB classifier on the Iris dataset (continuous features: sepal/petal lengths and widths) and evaluate its performance with a full classification report.

In [ ]:
# 1. Load Iris Dataset
iris = load_iris()
X_iris, y_iris = iris.data, iris.target

# 2. Train-Test Split
X_train, X_test, y_train, y_test = train_test_split(X_iris, y_iris, test_size=0.3, random_state=42)

# 3. Train Gaussian Naive Bayes
gnb = GaussianNB()
gnb.fit(X_train, y_train)

# 4. Predict
y_pred = gnb.predict(X_test)

# 5. Evaluation
print("Gaussian Naive Bayes - Iris Dataset")
print("="*55)
print(f"Accuracy: {accuracy_score(y_test, y_pred):.4f}")
print()
print("Classification Report:")
print(classification_report(y_test, y_pred, target_names=iris.target_names))

In [ ]:
# Inspect the learned parameters (mean and variance per class per feature)
print("Learned Parameters (Mean per class per feature):")
print("="*70)
feature_names = iris.feature_names
class_names = iris.target_names

mean_df = pd.DataFrame(gnb.theta_, columns=feature_names, index=class_names)
var_df = pd.DataFrame(gnb.var_, columns=feature_names, index=class_names)

print("\nClass Means (mu):")
print(mean_df.round(4))
print("\nClass Variances (sigma^2):")
print(var_df.round(4))
print(f"\nClass Priors: {dict(zip(class_names, gnb.class_prior_.round(4)))}")

### Visualizing Gaussian Distributions per Feature

Let's visualize how the Gaussian NB model sees each feature's distribution for each class. This shows the learned $\mu$ and $\sigma$ for each class-feature pair.

In [ ]:
from scipy.stats import norm

fig, axes = plt.subplots(2, 2, figsize=(14, 10))
colors = ['#e74c3c', '#2ecc71', '#3498db']

for idx, ax in enumerate(axes.flatten()):
    feature_col = X_iris[:, idx]
    x_range = np.linspace(feature_col.min() - 1, feature_col.max() + 1, 300)
    
    for k in range(3):
        mu = gnb.theta_[k, idx]
        sigma = np.sqrt(gnb.var_[k, idx])
        pdf_vals = norm.pdf(x_range, mu, sigma)
        ax.plot(x_range, pdf_vals, color=colors[k], linewidth=2.5, label=f'{class_names[k]} ($\\mu$={mu:.2f})')
        ax.fill_between(x_range, pdf_vals, alpha=0.15, color=colors[k])
    
    ax.set_title(f'{feature_names[idx]}', fontsize=12, fontweight='bold')
    ax.set_xlabel('Feature Value')
    ax.set_ylabel('Probability Density')
    ax.legend(fontsize=9, frameon=True, facecolor='white')

plt.suptitle('Gaussian NB: Learned Probability Distributions per Feature per Class', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

## 10. Demo B: Multinomial Naive Bayes for Text Classification

Multinomial NB is the gold standard for text classification. We will build a simple **spam detection** classifier from scratch using raw text data and Scikit-Learn's text processing pipeline.

In [ ]:
# 1. Create a small sample text corpus
emails = [
    "Win a free iPhone now click here",
    "Congratulations you have won a lottery prize",
    "Free money and gifts click now",
    "Claim your reward now before it expires",
    "You are selected for a cash prize click below",
    "Hey can we schedule a meeting tomorrow",
    "Please find the attached project report",
    "Let me know your availability for the call",
    "The quarterly budget review is next Monday",
    "Team lunch at noon today everyone is invited",
    "Hi please review the pull request when free",
    "Urgent deal buy now get 90 percent discount",
    "Special offer limited time only buy now",
    "Can you send me the updated spreadsheet",
    "Your order has been shipped tracking inside"
]
labels = [1, 1, 1, 1, 1, 0, 0, 0, 0, 0, 0, 1, 1, 0, 0]  # 1 = Spam, 0 = Not Spam

# 2. Convert text to word count features using CountVectorizer (Bag of Words)
vectorizer = CountVectorizer()
X_text = vectorizer.fit_transform(emails)

print(f"Vocabulary Size: {len(vectorizer.get_feature_names_out())} words")
print(f"Feature Matrix Shape: {X_text.shape} (samples x words)")
print(f"\nSample vocabulary: {vectorizer.get_feature_names_out()[:15]}")

In [ ]:
# 3. Train Multinomial Naive Bayes (with Laplace smoothing alpha=1.0)
mnb = MultinomialNB(alpha=1.0)
mnb.fit(X_text, labels)

# 4. Test on new unseen emails
test_emails = [
    "Free gift waiting for you click now",
    "Please review the attached document",
    "You won a million dollars claim now",
    "Meeting rescheduled to Friday morning"
]

X_test_text = vectorizer.transform(test_emails)
predictions = mnb.predict(X_test_text)
probabilities = mnb.predict_proba(X_test_text)

print("Multinomial Naive Bayes - Spam Detection")
print("="*65)
for email, pred, prob in zip(test_emails, predictions, probabilities):
    label = "SPAM" if pred == 1 else "NOT SPAM"
    print(f"Email: \"{email}\"")
    print(f"  -> Prediction: {label} | P(Not Spam): {prob[0]:.4f} | P(Spam): {prob[1]:.4f}")
    print()

## 11. Demo C: Bernoulli Naive Bayes

Let's compare Bernoulli NB with Multinomial NB on the same text data. Bernoulli NB binarizes features (presence vs absence of words) instead of using counts.

In [ ]:
# 1. Bernoulli NB (automatically binarizes input via binarize parameter)
bnb = BernoulliNB(alpha=1.0, binarize=0.0)  # binarize=0.0 means any count > 0 becomes 1
bnb.fit(X_text, labels)

# 2. Predictions
bnb_preds = bnb.predict(X_test_text)
bnb_probs = bnb.predict_proba(X_test_text)

# 3. Compare side by side
print("Comparison: Multinomial NB vs Bernoulli NB")
print("="*75)
print(f"{'Email':<45} | {'MultinomialNB':<15} | {'BernoulliNB':<15}")
print("="*75)
for email, m_pred, b_pred in zip(test_emails, predictions, bnb_preds):
    m_label = "SPAM" if m_pred == 1 else "NOT SPAM"
    b_label = "SPAM" if b_pred == 1 else "NOT SPAM"
    print(f"{email:<45} | {m_label:<15} | {b_label:<15}")
print("="*75)

## 12. Demo D: Confusion Matrix Heatmap & Model Comparison

Let's train all three Naive Bayes variants on the Iris dataset and compare their performance visually with confusion matrix heatmaps.

In [ ]:
# Train all three variants on Iris dataset
# Note: MultinomialNB requires non-negative features, so we use MinMaxScaler
from sklearn.preprocessing import MinMaxScaler

scaler = MinMaxScaler()
X_train_mm = scaler.fit_transform(X_train)
X_test_mm = scaler.transform(X_test)

models = {
    'GaussianNB': GaussianNB().fit(X_train, y_train),
    'MultinomialNB': MultinomialNB(alpha=1.0).fit(X_train_mm, y_train),
    'BernoulliNB': BernoulliNB(alpha=1.0).fit(X_train_mm, y_train)
}

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

for ax, (name, model) in zip(axes, models.items()):
    if name == 'GaussianNB':
        preds = model.predict(X_test)
    else:
        preds = model.predict(X_test_mm)
    
    cm = confusion_matrix(y_test, preds)
    acc = accuracy_score(y_test, preds)
    
    sns.heatmap(cm, annot=True, fmt='d', cmap='Purples', 
                xticklabels=iris.target_names, yticklabels=iris.target_names, ax=ax)
    ax.set_title(f'{name}\nAccuracy: {acc:.4f}', fontsize=12, fontweight='bold')
    ax.set_ylabel('Actual')
    ax.set_xlabel('Predicted')

plt.suptitle('Confusion Matrix Comparison: Three Naive Bayes Variants on Iris', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

## Summary Checklist for Naive Bayes

1. **Foundation is Bayes' Theorem:** $P(C|X) \propto P(X|C) \cdot P(C)$. We compute the posterior by combining the prior and the likelihood.
2. **The Naive Assumption:** Features are conditionally independent given the class. This converts a joint probability into a product of marginals: $P(X|C_k) = \prod P(x_j|C_k)$.
3. **Choose the Right Variant:**
   - **GaussianNB** for continuous features.
   - **MultinomialNB** for word counts/frequencies (text classification).
   - **BernoulliNB** for binary presence/absence features.
4. **Apply Laplace Smoothing ($\alpha$):** Always use `alpha >= 1` (default) to prevent zero probabilities from destroying posterior estimates.
5. **Use Log-Space for Computation:** Implementations use $\log P(C_k) + \sum \log P(x_j|C_k)$ to avoid floating-point underflow.
6. **Ideal for Text & High-Dimensional Sparse Data:** Naive Bayes is the go-to baseline for NLP classification tasks (spam, sentiment, topic detection).
7. **Fast Training & Prediction:** No iterative optimization — training is pure statistical counting. Use as a first baseline before trying complex models.